# Comprehensive Global Stock Market Research
## Factor-Based Analysis Across 5 Major Markets

**Markets Covered**: China, Europe, Japan, Korea, United States

**Analysis Components**
1. Factor Performance Analysis
2. Sector Analysis
3. Cross-Country Comparison
4. Market Interpretation & Insights

---
## Setup and Data Loading

In [ ]:
import configparser

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from openai import http_client

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palette for countries
COUNTRY_COLORS = {
    'china': '#E63946',
    'europe': '#457B9D',
    'japan': '#2A9D8F',
    'korea': '#E9C46A',
    'us': '#264653'
}

# Factor colors
FACTOR_COLORS = {
    'BM': '#E63946',
    'HML': '#F4A261',
    'SMB': '#2A9D8F',
    'MOM': '#264653',
    'OP': '#8338EC',
    'INV': '#06D6A0',
    'REV': '#118AB2'
}

# Base path
BASE_PATH = Path('.')
COUNTRIES = ['us', 'europe', 'japan', 'china', 'korea']

print("Setup complete. Loading data...")

In [ ]:
# GICS Sector mapping
SECTOR_NAMES = {
    '10': 'Energy',
    '15': 'Materials',
    '20': 'Industrials',
    '25': 'Consumer Disc.',
    '30': 'Consumer Staples',
    '35': 'Health Care',
    '40': 'Financials',
    '45': 'Info Tech',
    '50': 'Comm. Services',
    '55': 'Utilities',
    '60': 'Real Estate'
}

In [ ]:
# Data containers
data = {
    'total_ER': {}, # whole univers's equity expected returns
    'beta': {}, # whole universe's equity factor exposure
    'factor_hs': {}, # historical factor returns
    'factor_hs_clipped': {}, # clipped historical factor returns(recent 52 weeks, normalized to 100)
    'factor_ts': {}, # historical factor expected returns
    'factor_ts_clipped': {}, # clipped historical factor expected returns(recent 52 weeks, normalized to 100)
    'index_ts': {}, # historical index expected returns
    'index_ts_clipped': {}, # clipped historical index expected returns(recent 52 weeks, normalized to 100)
    'sector_ts': {}, # historical sector expected returns
    'sector_ts_clipped': {}, # clipped historical sector expected returns(recent 52 weeks, normalized to 100)
    'industry_ts' : {}, # historical industry expected returns
    'industry_ts_clipped': {}, # clipped historical industry expected returns(recent 52 weeks, normalized to 100)
    'factor_ER': {}, # factor expected returns for most recent month
    'index_ER': {}, # index expected returns for most recent month
    'sector_ER': {}, # sector expected returns for most recent month
    'industry_ER': {}, # industry expected returns for most recent month
    'size_ER': {}, # size quintile expected returns for most recent month
    'top_bot': {}, # top and bottom 10 stocks by expected return
}

def clip_ts(df, periods=52):
    """Clip time series to the last 'periods' entries and normalize to 100"""
    clipped = df[-periods:]
    clipped = (clipped / clipped.iloc[0]) * 100
    return clipped

# Load all data
for country in COUNTRIES:
    country_path = BASE_PATH / country

    # Load each file type
    data['total_ER'][country] = pd.read_csv(country_path / 'total_ER.csv', index_col=0, dtype={'sector':str})
    data['beta'][country] = pd.read_csv(country_path / 'beta.csv', index_col=0, dtype={'sector':str, 'industry':str})

    # historical time series of factor index
    data['factor_hs'][country] = pd.read_csv(country_path / 'factor_hs.csv', index_col=0, parse_dates=True)
    data['factor_hs_clipped'][country] = clip_ts(data['factor_hs'][country])
    # time series of factor index's expected return index
    data['factor_ts'][country] = pd.read_csv(country_path / 'factor_ts.csv', index_col=0, parse_dates=True)
    data['factor_ts_clipped'][country] = clip_ts(data['factor_ts'][country])
    # time series of country index's expected return index
    data['index_ts'][country] = pd.read_csv(country_path / 'index_ts.csv', index_col=0, parse_dates=True)
    data['index_ts_clipped'][country] = clip_ts(data['index_ts'][country])
    # time series of sector index's expected return index
    data['sector_ts'][country] = pd.read_csv(country_path / 'sector_ts.csv', index_col=0, parse_dates=True)
    data['sector_ts_clipped'][country] = clip_ts(data['sector_ts'][country])
    # time series of industry index's expected return index
    data['industry_ts'][country] = pd.read_csv(country_path / 'industry_ts.csv', index_col=0, parse_dates=True)
    data['industry_ts_clipped'][country] = clip_ts(data['industry_ts'][country])

    data['factor_ER'][country] = pd.read_csv(country_path / 'factor_ER.csv')
    data['index_ER'][country] = pd.read_csv(country_path / 'index_ER.csv')
    data['sector_ER'][country] = pd.read_csv(country_path / 'sector_ER.csv', dtype={'sector':str})
    data['industry_ER'][country] = pd.read_csv(country_path / 'industry_ER.csv', dtype={'industry':str})

    data['size_ER'][country] = pd.read_csv(country_path / 'size_ER.csv')
    data['top_bot'][country] = pd.read_csv(country_path / 'ERtop+bot10.csv', dtype={'sector':str})

print("Data loaded successfully!")
print(f"\nCountries: {COUNTRIES}")
print(f"\nStock counts by market:")
for country in COUNTRIES:
    print(f"  {country.upper()}: {len(data['total_ER'][country]):,} stocks")

In [ ]:
# NORMALIZATION FUNCTIONS FOR CROSS-COUNTRY COMPARISON
# Since factor models have different scales across countries, we need to
# normalize values before making cross-country comparisons.

def zscore_normalize(series):
    """Convert to z-scores (mean=0, std=1)"""
    return (series - series.mean()) / series.std()

def normalize(er, ts, er_ind:str, er_col:str, ts_col:str=None):
    """Normalize ER values using z-score based on time series data"""
    er_with_index = er.set_index(er_ind, inplace=False)
    er_val = er_with_index[er_col] / 100
    if ts_col is not None:
        ts_er = ts.filter(like=ts_col, axis=1)
        ts_er.columns = ts_er.columns.str.replace(ts_col, '')
    else:
        ts_er = ts
    ts_mean = ts_er.mean()
    ts_std = ts_er.std()

    data = pd.concat([er_val, ts_mean, ts_std], axis=1)
    data.columns = ['ER', 'Mean', 'Std']
    zscore = (data['ER'] - data['Mean']) / data['Std']
    zscore.name = 'ER_zscore'
    result = pd.concat([er_with_index, zscore], axis=1).dropna()

    return result

# Create normalized versions of key data
print("\n" + "="*70)
print("CREATING NORMALIZED DATA FOR CROSS-COUNTRY COMPARISON")
print("="*70)

# Normalize ER data for each country
for country in COUNTRIES:
    # Normalize equity ERs
    total_er = data['total_ER'][country].copy()
    if 'ER' in total_er.columns:
        total_er['ER_zscore'] = zscore_normalize(total_er['ER'])
    data['total_ER'][country] = total_er

    # Normalize index ERs
    er = data['index_ER'][country].copy()
    ts = data['index_ts'][country].copy()

    data['index_ER'][country] = normalize(er, ts, 'index', 'wER', '_ER')

    # Normalize sector ERs
    er = data['sector_ER'][country].copy()
    ts = data['sector_ts'][country].copy()

    data['sector_ER'][country] = normalize(er, ts, 'sector', 'wER', '_ER')

    # Normalize industry ERs
    er = data['industry_ER'][country].copy()
    ts = data['industry_ts'][country].copy()

    data['industry_ER'][country] = normalize(er, ts, 'industry', 'wER', '_ER')

    # Normalize factor returns (relative strength within country)
    er = data['factor_ER'][country].copy()
    ts = data['factor_ts'][country].copy()
    data['factor_ER'][country] = normalize(er, ts, 'factor', 'ER', '_ER')

print("Normalization methods applied:")
print("  • Z-Score: Standardized (mean=0, std=1) - good for relative comparisons")
print("  • Note: Use normalized values for cross-country comparisons")

In [ ]:
# ============================================================
# LLM Commentary Helper  (web search + persistent conversation)
# Requires: pip install openai
# Set environment variable: OPENAI_API_KEY=your-key
# ============================================================
import os
import datetime
from IPython.display import display, Markdown

OPENAI_MODEL = "gpt-5.6-terra"  # change to "gpt-4o" for higher quality

_conversation_history = []

_SYSTEM_MESSAGE = {
    "role": "system",
    "content": (
        "당신은 글로벌 대형 기관투자자를 대상으로 미국, 유럽, 일본, 중국, 한국 시장의 "
        "멀티팩터 포트폴리오 전략을 수립하는 '뉴스 통합형 시니어 퀀트 전략가(News-Integrated Chief Quant Strategist)'입니다.\n\n"
        "【핵심 분석 원칙】\n"
        "1. 최신 뉴스 우선: 반드시 현재 날짜 기준 최근 2~4주 이내의 실제 시장 뉴스, 거시경제 이벤트, 정책 발표, 기업 공시를 먼저 검색하고 이를 분석의 중심으로 삼으십시오.\n"
        "2. 팩터 데이터는 보조 근거: 제공된 Z-스코어, 방향성(상승/하락) 등 팩터 데이터는 뉴스 해석을 뒷받침하는 정량적 확인 도구로만 활용하십시오. 수치를 직접 나열하는 것을 금지합니다.\n"
        "3. 이벤트-드리븐 내러티브: 모든 분석은 '어떤 사건/정책/테마가 이 팩터·섹터·종목을 움직이는가'라는 질문에 답하는 형식으로 서술하십시오.\n"
        "4. Bottom-line Up Front: 가장 중요한 뉴스 기반 투자 시그널부터 시작하고, 수치는 괄호 안에 간략히 표기하십시오.\n"
        "5. 격식체 한국어: 투자위원회(IC) 제출용 보고서에 부합하는 명사형 어미(~함, ~임, ~판단, ~권고)를 사용하십시오.\n"
        "6. 최신성 보장: 검색 시 반드시 현재 연도·월을 포함한 쿼리를 사용하여 최신 뉴스만 인용하십시오."
        " [팩터 종류] \n"
        "1. Beta(BM): 주식시장과의 민감도. 일반적으로 시가총액이 높을수록 1에 가까우므로 대형주와 유사"
        "2. Value(HML): 가치(B/P)가 낮은 저평가 종목"
        "3. Size(SMB): 시가총액이 낮은 소형주"
        "4. Investment(INV): 투자(총자산 증가율이 높은 기업)를 공격적으로 하는 기업. 설비투자, CAPEX를 공격적으로 늘리는 기업으로 생각할 수 있음"
        "5. Profitability(OP): ROE가 높은 고이익 기업"
        "6. Momentum(MOM): 최근 주가 상승 탄력(12m-1m)이 높은 기업"
        "7. Reversal(REV): 최근 하락한 주가가 크게 하락한 종목일수록 평균회귀 효과로 주가 상승 기대"
    )
}

def reset_conversation():
    global _conversation_history
    _conversation_history = []
    print("대화 기록이 초기화되었습니다.")

def llm_explain(data_str: str, prompt: str, use_search: bool = True) -> str:
    global _conversation_history
    try:
        from openai import OpenAI
        import httpx
        key = os.getenv('OPENAI_API_KEY', '')
        if not key:
            return "_LLM 해설을 사용할 수 없습니다 — `OPENAI_API_KEY` 환경 변수를 설정하세요._"

        client = OpenAI(api_key=key, http_client=httpx.Client(verify=False))
        today = datetime.date.today().isoformat()
        date_preamble = f"[오늘 날짜: {today}. 최근 2~4주 이내의 최신 뉴스를 검색하여 분석에 활용하십시오.]\n\n"
        user_message = {
            "role": "user",
            "content": date_preamble + prompt + "\n\nData:\n" + data_str
        }
        messages = [_SYSTEM_MESSAGE] + _conversation_history + [user_message]
        kwargs = dict(model=OPENAI_MODEL, input=messages)
        if use_search:
            kwargs["tools"] = [{"type": "web_search_preview"}]
        response = client.responses.create(**kwargs)
        reply = response.output_text
        _conversation_history.append(user_message)
        _conversation_history.append({"role": "assistant", "content": reply})
        return reply
    except ImportError:
        return "_`openai` 패키지가 설치되지 않았습니다 — `pip install openai`를 실행하세요._"
    except Exception as e:
        return f"_LLM 오류: {e}_"

def show_explanation(data_str: str, prompt: str, title: str = "분석 해설", use_search: bool = True):
    text = llm_explain(data_str, prompt, use_search=use_search)
    display(Markdown(f"---\n### 💬 {title}\n\n{text}\n\n---"))

reset_conversation()
print(f"LLM 헬퍼 로드 완료. 모델: {OPENAI_MODEL}")
print("대화 메모리: 활성화 (초기화하려면 reset_conversation() 호출)")
print("웹 검색: 활성화 (최신 뉴스 기반 분석)")
print("API 키:", "✓ 설정됨" if os.getenv('OPENAI_API_KEY') else "✗ 미설정 (OPENAI_API_KEY 환경 변수를 설정하세요)")

---
## Section 1: 과거 관찰 — 팩터 역사적 행동

과거 팩터 시계열을 분석하여 현재 시장 국면을 진단합니다.

### Factor Time Series Performance

In [ ]:
# Plot factor time series for each country
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for idx, country in enumerate(COUNTRIES):
    ax = axes[idx]
    factor_ts = data['factor_hs_clipped'][country]

    for col in factor_ts.columns:
        if col in FACTOR_COLORS:
            ax.plot(factor_ts.index, factor_ts[col], label=col,
                   color=FACTOR_COLORS.get(col, 'gray'), linewidth=1.5)

    ax.axhline(y=100, color='black', linestyle='--', alpha=0.3)
    ax.set_title(f'{country.upper()} - Factor Performance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Index (Base=100)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide the 6th subplot
axes[5].axis('off')

plt.suptitle('Factor Performance Time Series by Country', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate factor statistics
def calculate_factor_stats(factor_ts, factor_hs):
    """Calculate comprehensive factor statistics"""
    returns = factor_ts.pct_change().dropna()

    stats = pd.DataFrame({
        'Total Return (%)': ((factor_ts.iloc[-1] / factor_ts.iloc[0]) - 1) * 100,
        'Ann. Return (%)': returns.mean() * 52 * 100,
        'Ann. Vol (%)': returns.std() * np.sqrt(52) * 100,
        'Sharpe Ratio': (returns.mean() * 52) / (returns.std() * np.sqrt(52)),
        'Max Drawdown (%)': ((factor_ts / factor_ts.cummax()) - 1).min() * 100,
        'Max Drawdown(total) (%)': ((factor_hs / factor_hs.cummax()) - 1).min() * 100,
        'Skewness': returns.skew(),
        'Kurtosis': returns.kurtosis()
    })
    return stats.round(3)

# Display factor stats for each country
for country in COUNTRIES:
    print(f"\n{'='*70}")
    print(f"{country.upper()} - FACTOR STATISTICS")
    print("="*70)
    stats = calculate_factor_stats(data['factor_hs_clipped'][country], clip_ts(data['factor_hs'][country], periods=52*5))
    display(stats.style.format('{:.1f}').background_gradient(cmap='RdYlGn', subset=['Sharpe Ratio'], axis=0))

### Factor Correlation Analysis

In [ ]:
# Factor correlation matrices
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for idx, country in enumerate(COUNTRIES):
    ax = axes[idx]
    factor_returns = data['factor_hs_clipped'][country].pct_change().dropna()
    corr = factor_returns.corr()

    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, ax=ax, vmin=-1, vmax=1,
                square=True, linewidths=0.5)
    ax.set_title(f'{country.upper()} - Factor Correlations', fontsize=12, fontweight='bold')

axes[5].axis('off')
plt.suptitle('Factor Correlation Matrices by Country', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Section 1 LLM: 과거 관찰 — 팩터 역사적 동인 ===
_hist_txt = f"HISTORICAL ANALYSIS — Factor Direction Summary (week ending approx. {datetime.date.today().isoformat()}\n\n"

for _c in COUNTRIES:
    _hs = data['factor_hs_clipped'][_c]
    _ret4   = (_hs.iloc[-1] / _hs.iloc[-5]  - 1) * 100 if len(_hs) >= 5  else pd.Series(dtype=float)
    _prior4 = (_hs.iloc[-5] / _hs.iloc[-9] - 1) * 100  if len(_hs) >= 9  else pd.Series(dtype=float)
    _accel  = (_ret4 - _prior4).dropna().sort_values(ascending=False)
    _top3   = list(_accel.head(3).index)
    _bot3   = list(_accel.tail(3).index)
    _hist_txt += f"{_c.upper()}: 가속 팩터={', '.join(_top3)} | 감속 팩터={', '.join(_bot3)}\n"

_hist_prompt = (
    "역할: 글로벌 퀀트 전략가. 이 섹션이 보고서의 첫 분석 단계입니다.\n\n"
    "제공된 국가별 팩터 가속/감속 방향 요약을 바탕으로 각 시장의 최신 거시경제 뉴스를 검색하십시오.\n\n"
    "검색 지시: 각 국가별로 '[국가명] 주식시장 팩터 [현재연월]' 쿼리로 최신 이벤트를 검색하십시오.\n\n"
    "출력 구조:\n"
    "1. 국가별 팩터 동향 관찰: 각 시장에서 팩터 흐름을 주도한 뉴스 이벤트/정책/테마. 팩터 방향은 괄호 안 간략 표기.\n"
    "2. 글로벌 공통 스타일 패턴: 여러 시장에서 동시에 관찰되는 스타일 로테이션과 배경 이벤트.\n"
    "3. 주목할 이상 신호: 글로벌 트렌드와 반하는 특이 시장과 그 이유.\n\n"
    "분량: A4 0.5페이지. 수치 나열 금지. 뉴스 내러티브 우선.\n"
    "※ 이 관찰은 다음 섹션(미래 신호)의 출발점으로 이어집니다."
)

show_explanation(_hist_txt, _hist_prompt, title="Section 1: 과거 관찰 — 팩터 역사적 동인 분석")


---
## Section 2: 미래 신호 — 모델의 포워드 뷰

기대수익률(Expected Return)을 통해 모델이 예측하는 미래를 확인합니다.

### Market Expected Returns

In [ ]:
# Market-level expected returns comparison - WITH NORMALIZATION CONTEXT
print("="*70)
print("MARKET-LEVEL ANALYSIS")
print("="*70)
print("\n⚠️  IMPORTANT: Market ER values are NOT directly comparable across countries")
print("    because factor models have different scales/calibrations.\n")

# Raw market ERs
market_er = {}
for country in COUNTRIES:
    wmean = data['index_ER'][country]['ER_zscore'].iloc[0]
    market_er[country.upper()] = wmean

market_er_df = pd.DataFrame([market_er], index=['Weighted Mean ER (%)']).T

print("[1] MARKET ER - (z-score scales)")
print("-"*70)
display(market_er_df.style.format('{:.2f}'))

# Calculate comparable metrics using within-country distribution
print("\n[2] CROSS-COUNTRY COMPARABLE METRICS")
print("-"*70)

comparable_metrics = []
for country in COUNTRIES:
    total_stock_er = data['total_ER'][country]
    index_total_history_er = data['index_ts'][country]['index_ER']
    index_er = data['index_ER'][country]['wER'].iloc[0]

    # What percentile is the market ER within its own distribution?
    market_percentile = (index_er/100 > index_total_history_er).mean() * 100

    # Z-score of market ER within country
    market_zscore = data['index_ER'][country]['ER_zscore'].iloc[0]

    # Stock-wise
    er_values = total_stock_er['ER'].dropna()
    sh_values = total_stock_er['Sharpe'].dropna()

    # Percentage of stocks with positive ER
    pct_positive = (er_values > 0).mean() * 100

    # Spread between top and bottom quintile
    er_spread = er_values.quantile(0.8) - er_values.quantile(0.2)

    # Sharpe-like metric: mean ER / std ER
    er_sharpe = sh_values.mean()

    comparable_metrics.append({
        'Country': country.upper(),
        'Market ER Percentile': market_percentile,
        'Market ER Z-Score': market_zscore,
        '% Stocks Positive ER': pct_positive,
        'ER Dispersion (Q80-Q20)': er_spread,
        'ER Sharpe (mean/std)': er_sharpe
    })

comparable_df = pd.DataFrame(comparable_metrics).set_index('Country')
display(comparable_df.style.format({
    'Market ER Percentile': '{:.1f}',
    'Market ER Z-Score': '{:.2f}',
    '% Stocks Positive ER': '{:.1f}%',
    'ER Dispersion (Q80-Q20)': '{:.2f}',
    'ER Sharpe (mean/std)': '{:.2f}'
}).background_gradient(cmap='RdYlGn', axis=0))

print("\nInterpretation Guide:")
print("  • Market ER Percentile: Higher = More strong signal for the market relative to historical periods")
print("  • % Stocks Positive ER: Higher = More bullish signal for the market")
print("  • ER Sharpe: Higher = More consistent positive signal across stocks")

# Bar chart of comparable metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Market Percentile
ax = axes[0]
colors = [COUNTRY_COLORS[c.lower()] for c in comparable_df.index]
comparable_df['Market ER Percentile'].plot(kind='bar', ax=ax, color=colors)
ax.set_title('Market ER Percentile', fontweight='bold')
ax.set_ylabel('Percentile')
ax.axhline(y=50, color='black', linestyle='--', alpha=0.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Plot 2: % Positive ER
ax = axes[1]
comparable_df['% Stocks Positive ER'].plot(kind='bar', ax=ax, color=colors)
ax.set_title('% of Stocks with Positive ER', fontweight='bold')
ax.set_ylabel('Percentage')
ax.axhline(y=50, color='black', linestyle='--', alpha=0.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Plot 3: ER Sharpe
ax = axes[2]
comparable_df['ER Sharpe (mean/std)'].plot(kind='bar', ax=ax, color=colors)
ax.set_title('ER Sharpe (Signal Strength)', fontweight='bold')
ax.set_ylabel('Mean/Std Ratio')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('Cross-Country Comparable Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Factor Expected Returns

In [ ]:
# Create factor returns comparison table - RAW and NORMALIZED
print("="*70)
print("FACTOR RETURNS BY COUNTRY")
print("="*70)
print("    Use Z-Score for valid cross-country comparisons.\n")

# Z-Score normalized (within each country)
factor_zscore_df = pd.DataFrame()
for country in COUNTRIES:
    fr = data['factor_ER'][country]['ER_zscore']
    factor_zscore_df[country.upper()] = fr
factor_zscore_df = factor_zscore_df.T

print("\n[2] Z-SCORE NORMALIZED - Comparable across countries (0 = country average)")
print("-"*70)
display(factor_zscore_df.style.format('{:.2f}').background_gradient(cmap='RdYlGn', axis=None))

# Cross-country factor comparison using z-scores
print("\n[4] CROSS-COUNTRY FACTOR STRENGTH (Z-Score Average)")
print("-"*70)
print("Which factors are consistently strong/weak across all markets:")
factor_strength = factor_zscore_df.mean().sort_values(ascending=False)
for factor, zscore in factor_strength.items():
    strength = "STRONG ↑" if zscore > 0.5 else ("WEAK ↓" if zscore < -0.5 else "NEUTRAL →")
    bar = "█" * int(abs(zscore) * 10)
    print(f"  {factor:5}: {zscore:+.2f} {strength} {bar}")

### Factor Premium Comparison

In [ ]:
# Factor premium comparison across countries
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

factors = ['BM', 'HML', 'SMB', 'MOM', 'OP', 'INV']

for idx, factor in enumerate(factors):
    ax = axes[idx]
    factor_vals = []

    for country in COUNTRIES:
        fr = data['factor_ER'][country]['ER_zscore']
        if factor in fr.index:
            factor_vals.append(fr.loc[factor])
        else:
            factor_vals.append(0)

    colors = [COUNTRY_COLORS[c] for c in COUNTRIES]
    bars = ax.bar([c.upper() for c in COUNTRIES], factor_vals, color=colors)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax.set_title(f'{factor} Factor Premium', fontsize=12, fontweight='bold')
    ax.set_ylabel('Factor Return (%)')

    # Add value labels
    for bar, val in zip(bars, factor_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Factor Premiums Across Countries', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Sector Expected Returns

In [ ]:
# Sector Expected Returns comparison - RAW and NORMALIZED
print("="*90)
print("SECTOR EXPECTED RETURNS BY COUNTRY")
print("="*90)
print("\n⚠️  NOTE: Raw ER values have different scales across countries.")
print("    Use Z-Score for valid cross-country sector comparisons.\n")

# Z-Score normalized sector ERs
sector_zscore_comparison = pd.DataFrame()
for country in COUNTRIES:
    ind_er = data['sector_ER'][country].copy()
    ind_er = ind_er.set_index('Sector')
    sector_zscore_comparison[country.upper()] = ind_er['ER_zscore']

print("\n[2] Z-SCORE NORMALIZED - Comparable (0 = average sector in country)")
print("-"*70)
display(sector_zscore_comparison.style.format('{:.2f}').background_gradient(cmap='RdYlGn', axis=None))

# Cross-country sector analysis
print("\n[4] GLOBAL SECTOR ATTRACTIVENESS (Average Z-Score across countries)")
print("-"*70)
sector_global_strength = sector_zscore_comparison.mean(axis=1).sort_values(ascending=False)
for sector, zscore in sector_global_strength.items():
    strength = "OVERWEIGHT" if zscore > 0.3 else ("UNDERWEIGHT" if zscore < -0.3 else "NEUTRAL")
    bar = "█" * int(abs(zscore) * 10)
    print(f"  {str(sector):25}: {zscore:+.2f} {strength} {bar}")

In [ ]:
# === Section 2 LLM: 미래 신호 — 모델의 포워드 뷰 ===
_forward_txt = f"FORWARD SIGNALS — Factor & Market Expected Returns (as of {datetime.date.today().isoformat()}\n\n"

_factors_list = ['BM', 'HML', 'SMB', 'MOM', 'OP', 'INV']

_forward_txt += "글로벌 팩터 기대수익률 방향 (4주 기준, 국가별 상승↑/하락↓):\n"
for _f in _factors_list:
    _col = f"{_f}_ER"
    _dirs = []
    for _c in COUNTRIES:
        _fts = data['factor_ts'][_c]
        if _col in _fts.columns and len(_fts) >= 5:
            _chg = float(_fts[_col].iloc[-1]) - float(_fts[_col].iloc[-5])
            _dirs.append(f"{_c.upper()}:{'↑' if _chg > 0 else '↓'}")
        else:
            _dirs.append(f"{_c.upper()}:?")
    _up = sum(1 for d in _dirs if '↑' in d)
    _consensus = "글로벌 상승" if _up >= 4 else ("글로벌 하락" if _up <= 1 else "혼조")
    _forward_txt += f"  {_f}: {' '.join(_dirs)} → {_consensus}\n"

_forward_txt += "\n국가별 시장 ER 상황:\n"
for _c in COUNTRIES:
    _mz = data['index_ER'][_c]['ER_zscore'].iloc[0]
    _top = data['factor_ER'][_c]['ER_zscore'].idxmax()
    _bot = data['factor_ER'][_c]['ER_zscore'].idxmin()
    _forward_txt += f"  {_c.upper()}: 시장ER z={_mz:.2f} | 최강팩터={_top} | 최약팩터={_bot}\n"

_forward_txt += "\n글로벌 섹터 매력도 (상위3/하위3):\n"
_sec_z = {}
for _c in COUNTRIES:
    _s_df = data['sector_ER'][_c].copy()
    if 'Sector' not in _s_df.columns and 'sector' in _s_df.columns:
        _s_df['Sector'] = _s_df['sector'].astype(str).map(SECTOR_NAMES)
    for _, _row in _s_df.iterrows():
        _sec = _row.get('Sector', '')
        if _sec and pd.notna(_sec):
            _sec_z.setdefault(str(_sec), []).append(_row.get('ER_zscore', 0))
_sec_avg = {s: sum(v)/len(v) for s, v in _sec_z.items() if v}
_sec_sorted = sorted(_sec_avg.items(), key=lambda x: x[1], reverse=True)
_forward_txt += "  상위: " + ", ".join(s for s, _ in _sec_sorted[:3]) + "\n"
_forward_txt += "  하위: " + ", ".join(s for s, _ in _sec_sorted[-3:]) + "\n"

_forward_prompt = (
    "[이전 섹션에서 확인한 팩터 동향과 뉴스 드라이버를 이어받아 분석합니다.]\n\n"
    "역할: 글로벌 퀀트 전략가. 제공되는 데이터는 '기대수익률(Expected Return)' — "
    "모델이 예측하는 미래 팩터·시장·섹터 수익률입니다.\n\n"
    "검색 지시: 각 국가별로 '[국가명] 경제 전망 [현재연월]' 쿼리로 최신 포워드 뷰 뉴스를 검색하십시오.\n\n"
    "출력 구조:\n"
    "1. 과거 추세와의 연속성/단절: 앞서 관찰한 팩터 모멘텀이 기대수익률에서도 지속되는가, "
    "아니면 모델이 반전을 예고하는가? (국가별 비교)\n"
    "2. 국가별 미래 포지셔닝: 기대수익률 기준 가장 매력적/비매력적인 시장과 그 이유 (뉴스 근거 포함).\n"
    "3. 주목할 섹터 방향: 기대수익률 상위 섹터가 강세인 배경 이벤트와 그 지속 가능성.\n\n"
    "분량: A4 0.5페이지. 수치 나열 금지.\n"
    "※ 다음 섹션에서 이 분석을 바탕으로 글로벌 통합 테제를 선언합니다."
)

show_explanation(_forward_txt, _forward_prompt, title="Section 2: 미래 신호 — 모델의 포워드 뷰 분석")


---
## Section 3: 글로벌 테제 합성

과거 관찰과 미래 신호를 통합하여 하나의 시장 테제를 도출합니다.

In [ ]:
# === Section 3 LLM: 글로벌 테제 합성 (NEW) ===
_thesis_txt = (
    f"GLOBAL THESIS SYNTHESIS (as of {datetime.date.today().isoformat()}\n\n"
    "앞선 두 섹션에서 다음이 확인됨:\n"
    "  • Section 1: 국가별 팩터 가속/감속 동향 및 뉴스 드라이버\n"
    "  • Section 2: 팩터·시장·섹터 기대수익률의 포워드 신호\n"
    "이 두 관찰을 통합하여 글로벌 투자 테제를 도출하십시오.\n"
)

_thesis_prompt = (
    "[지금까지 두 섹션의 분석 — 과거 팩터 행동 관찰과 미래 기대수익률 신호 — 을 통합합니다.]\n\n"
    "역할: 수석 투자전략가(CIO). 이 섹션에서 공식 시장 테제를 선언합니다.\n\n"
    "출력 구조:\n"
    "1. 글로벌 시장 테제 (1-2문장): "
    ""현재 글로벌 시장은 [X 국면]에 있으며, [Y 팩터/섹터/스타일]이 유리한 환경이다. "
    "핵심 드라이버는 [Z 이벤트/정책/매크로]이다."\n"
    "2. 지역별 테제 강도: 이 테제가 가장 강하게 적용되는 시장 vs. 예외·이탈 시장 (이유 포함).\n"
    "3. 테제 유효 기간과 무효화 조건: 어떤 이벤트/지표가 발생하면 이 테제를 수정해야 하는가.\n\n"
    "분량: A4 0.5페이지.\n"
    "이 섹션은 보고서 전체의 중심 주장입니다 — 명확하고 단호하게 서술할 것. 수치 나열 금지."
)

show_explanation(_thesis_txt, _thesis_prompt, title="Section 3: 글로벌 테제 합성")


---
## Section 4: 섹터·팩터 함의

확정된 테제를 기반으로 섹터 배분 방향을 도출합니다.

### Sector Factor Exposures

In [ ]:
# Sector factor exposure heatmaps
factor_cols = ['beta_bm', 'beta_hml', 'beta_smb', 'beta_op', 'beta_inv', 'beta_mom']

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for idx, country in enumerate(COUNTRIES):
    ax = axes[idx]
    ind_er = data['sector_ER'][country].copy()

    # Get sector names as index
    if 'Sector' in ind_er.columns:
        ind_er = ind_er.set_index('Sector')
    elif 'industry' in ind_er.columns:
        ind_er['Sector'] = ind_er['industry'].astype(str).map(SECTOR_NAMES)
        ind_er = ind_er.set_index('Sector')

    # Get available factor columns
    available_factors = [col for col in factor_cols if col in ind_er.columns]

    if available_factors:
        plot_data = ind_er[available_factors].dropna(how='all')
        plot_data.columns = [c.replace('beta_', '').upper() for c in plot_data.columns]

        sns.heatmap(plot_data, annot=True, fmt='.1f', cmap='RdBu_r',
                    center=0, ax=ax, linewidths=0.5)
        ax.set_title(f'{country.upper()} - Sector Factor Exposures', fontsize=12, fontweight='bold')
        ax.set_xlabel('Factor')
        ax.set_ylabel('')
    else:
        ax.text(0.5, 0.5, 'No factor data available', ha='center', va='center')
        ax.set_title(f'{country.upper()}')

axes[5].axis('off')
plt.suptitle('Sector Factor Exposures by Country', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Section 4 LLM: 섹터·팩터 함의 ===
_beta_cols_map = {'BM': 'beta_bm', 'HML': 'beta_hml', 'SMB': 'beta_smb',
                  'OP': 'beta_op', 'INV': 'beta_inv', 'MOM': 'beta_mom'}

_sector_impl_txt = "FACTOR-SECTOR ALIGNMENT ANALYSIS\n\n"

_f_zscores = {}
for _f in ['BM', 'HML', 'SMB', 'MOM', 'OP', 'INV']:
    _vals = [data['factor_ER'][_c]['ER_zscore'].get(_f, 0) for _c in COUNTRIES]
    _f_zscores[_f] = sum(_vals) / len(_vals)

_strong = [f for f, z in _f_zscores.items() if z > 0.3]
_weak   = [f for f, z in _f_zscores.items() if z < -0.3]
_sector_impl_txt += f"현재 강세 팩터: {', '.join(_strong) if _strong else '없음'}\n"
_sector_impl_txt += f"현재 약세 팩터: {', '.join(_weak) if _weak else '없음'}\n\n"

_sector_impl_txt += "국가별/섹터별 팩터 정렬 점수 상세:\n"
for _c in COUNTRIES:
    _sector_impl_txt += f"  [{_c.upper()}]\n"
    _sec_df = data['industry_ER'][_c].copy()
    if 'Sector' in _sec_df.columns:
        _sec_df = _sec_df.set_index('Sector')
    elif 'industry' in _sec_df.columns:
        _sec_df = _sec_df.set_index('industry')
    for _sec in _sec_df.index:
        _factor_details = []
        for _f, _bcol in _beta_cols_map.items():
            if _bcol in _sec_df.columns and _f in _f_zscores:
                _bval = _sec_df.loc[_sec, _bcol]
                _factor_details.append(f"{_f}:{_bval:.2f}")
        _sector_impl_txt += f"    - {_sec}: {', '.join(_factor_details)}\n"
    _sector_impl_txt += "\n"

_sector_impl_prompt = (
    "[확정된 글로벌 테제를 기반으로 섹터 배분 함의를 도출합니다.]\n\n"
    "역할: 글로벌 팩터-섹터 전략가.\n\n"
    "수혜 섹터 = 강세 팩터에 높은 베타 노출 (팩터 프리미엄 수령), "
    "역풍 섹터 = 약세 팩터에 높은 베타 노출 (팩터 드래그).\n\n"
    "출력 구조:\n"
    "1. 테제 수혜 섹터 (표): 국가 | 섹터 | 수혜 근거 (팩터-섹터 정렬 + 뉴스 촉매) — 국가별 3개씩\n"
    "2. 테제 역풍 섹터 (표): 국가 | 섹터 | 역풍 근거 — 국가별 3개씩\n"
    "3. 글로벌 오버웨이트/언더웨이트 권고: 테제에 기반한 섹터 배분 방향 요약.\n\n"
    "수치 나열 금지. 섹터 배분 관점의 핵심만 서술.\n"
    "※ 다음 섹션에서 이 섹터 방향을 구체적인 종목으로 연결합니다."
)

show_explanation(_sector_impl_txt, _sector_impl_prompt,
                 title="Section 4: 섹터·팩터 함의 — 테제 기반 섹터 배분", use_search=True)


---
## Section 5: 국가별 시사점

글로벌 테제가 각 시장에서 어떻게 나타나는지, 그리고 그 안에서 가장 유망한 종목을 도출합니다.

### Top Equities by country

In [ ]:
# Top 10 stocks by Expected Return for each country
print("="*90)
print("TOP 10 STOCKS BY EXPECTED RETURN (Each Country)")
print("="*90)

for country in COUNTRIES:
    print(f"\n{'-'*70}")
    print(f"{country.upper()}")
    print("-"*70)

    top_bot = data['top_bot'][country]
    top_bot['Sector'] = top_bot['sector'].map(lambda x: SECTOR_NAMES[x], na_action='ignore')
    # Get top 10 (highest ER)
    if 'ER' in top_bot.columns:
        top10 = top_bot.nlargest(10, 'ER')[['instrument', 'Name', 'Sector', 'ER', 'Vol', 'Sharpe', 'mktval']]
        top10.columns = ['ISIN', 'Name', 'Sector', 'Expected Return(%)', 'Vol(%)', 'Sharpe(X)', 'Market Cap(Local mn)']
        top10 = top10.style.format({
            'Expected Return(%)': '{:.2f}',
            'Vol(%)': '{:.2f}',
            'Sharpe(X)': '{:.2f}',
            'Market Cap(Local mn)': '{:,.0f}',
        })
        display(top10)

In [ ]:
# === Section 5 LLM: 국가별 시사점 (per-country loop) ===
for _c in COUNTRIES:
    _td_txt = f"TOP-DOWN ANALYSIS: {_c.upper()} (as of {datetime.date.today().isoformat()}\n\n"

    _ind_z = data['industry_ER'][_c].copy()
    if 'industry' in _ind_z.columns:
        _ind_z = _ind_z.set_index('industry')
    if 'ER_zscore' in _ind_z.columns:
        _top5_ind = _ind_z['ER_zscore'].dropna().nlargest(5)
        _td_txt += "상위 5개 산업 (ER 강세 기준):\n"
        for _ind_code, _z in _top5_ind.items():
            _td_txt += f"  Industry code {_ind_code}\n"

    _tb = data['top_bot'][_c].copy()
    _tb['SectorName'] = _tb['sector'].map(lambda x: SECTOR_NAMES.get(str(x), str(x)))
    if 'ER' in _tb.columns:
        _top10 = _tb.nlargest(10, 'ER')[['Name', 'SectorName']]
        _td_txt += "\n상위 10개 종목 (종목명 | 섹터):\n"
        for _, _row in _top10.iterrows():
            _td_txt += f"  {_row['Name']} | {_row['SectorName']}\n"

    _td_prompt = (
        f"[확정된 글로벌 테제와 섹터 배분 방향을 기반으로 {_c.upper()} 시장 시사점을 도출합니다.]\n\n"
        f"역할: {_c.upper()} 시장 전문 애널리스트.\n\n"
        f"검색 지시: '[{_c}] [상위 산업명] [{datetime.date.today().strftime('%Y %B')}]' 쿼리로, "
        f"각 상위 종목명으로 '[종목명] {datetime.date.today().year} news' 형태로 검색하십시오.\n\n"
        "출력 구조 (탑다운):\n"
        f"1. 테제의 {_c.upper()} 적용: 글로벌 테제가 이 시장에서 어떻게 강화 또는 완화되는가? 시장 특수성 설명.\n"
        "2. 수혜 산업 테마: 테제에 정렬된 상위 산업과 최신 뉴스 촉매.\n"
        "3. 컨빅션 픽 3개: 산업 테마 + 종목 뉴스가 가장 정렬된 종목. 각 투자 근거 한 문장.\n\n"
        "분량: A4 0.3페이지. 뉴스 이벤트 중심. 한국어로 작성."
    )

    show_explanation(_td_txt, _td_prompt, title=f"Section 5: {_c.upper()} 시장 시사점")


---
## Section 6: 최종 통합 결론

투자위원회(IC) 제출용 최종 결론입니다.

In [ ]:
# === Section 6 LLM: 최종 통합 결론 (NEW) ===
_final_txt = (
    f"FINAL SYNTHESIS (as of {datetime.date.today().isoformat()}\n\n"
    "모든 섹션 분석 완료:\n"
    "  • Section 1: 과거 팩터 관찰 + 뉴스 드라이버\n"
    "  • Section 2: 미래 기대수익률 포워드 뷰\n"
    "  • Section 3: 글로벌 테제 선언\n"
    "  • Section 4: 섹터·팩터 배분 함의\n"
    "  • Section 5: 5개 국가별 컨빅션 픽\n"
    "이 모든 분석을 하나로 통합하십시오.\n"
)

_final_prompt = (
    "[모든 섹션의 분석 — 과거 관찰, 미래 신호, 글로벌 테제, 섹터 함의, 국가별 시사점 — 을 통합합니다.]\n\n"
    "역할: 수석 투자전략가(CIO). 투자위원회(IC) 제출용 최종 결론 페이지입니다.\n\n"
    "출력 구조:\n"
    "1. 확정 글로벌 테제 (1문장): 모든 분석이 수렴한 최종 시장 관점.\n"
    "2. 국가별 비중 방향 (표): 국가 | 비중 방향(오버/뉴트럴/언더) | 핵심 근거 1줄\n"
    "3. 최선호 섹터·팩터 (글로벌 상위 3개): 각 한 줄 근거.\n"
    "4. 고확신 종목 (글로벌 상위 5-10개): 종목명 | 국가 | 핵심 투자 테제 (한 문장)\n"
    "5. 핵심 리스크: 테제를 무효화할 수 있는 이벤트 3개.\n\n"
    "분량: A4 1페이지. 의사결정자가 이 페이지만 읽어도 포지션을 잡을 수 있어야 함."
)

show_explanation(_final_txt, _final_prompt, title="Section 6: 최종 통합 결론 — IC 제출용 투자 결론")


In [ ]:
!jupyter nbconvert --to html C:\Users\NHWM\PycharmProjects\Factor_Research\output\0.export\market_research.ipynb --no-input

In [ ]:
!pandoc C:\Users\NHWM\PycharmProjects\Factor_Research\output\0.export\market_research.html -o C:\Users\NHWM\PycharmProjects\Factor_Research\output\0.export\market_research.docx